In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
vanshikavmittal_fakeddit_dataset_path = kagglehub.dataset_download('vanshikavmittal/fakeddit-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vanshikavmittal/fakeddit-dataset")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd

# Load the TSV file properly
df_train = pd.read_csv('/kaggle/input/fakeddit-dataset/multimodal_only_samples/multimodal_train.tsv', sep='\t')
df_test = pd.read_csv('/kaggle/input/fakeddit-dataset/multimodal_only_samples/multimodal_test_public.tsv', sep='\t')

In [ ]:
df_selected_feture_train = df_train[["clean_title","image_url","2_way_label"]]
df_selected_feture_test = df_test[["clean_title","image_url","2_way_label"]]

In [ ]:
import os
os.makedirs("images", exist_ok=True)
os.makedirs("images/test", exist_ok=True)
os.makedirs("images/train", exist_ok=True)

In [ ]:
df_selected_feture_train.shape

In [ ]:
label_counts = df_selected_feture_train['2_way_label'].value_counts()
print("Original class distribution:\n", label_counts)

In [ ]:
label_counts = df_selected_feture_test['2_way_label'].value_counts()
print("Original class distribution:\n", label_counts)

In [ ]:
import pandas as pd
import os

# Assume these exist already and are filtered
# df_train, df_val, df_test

# Combine just the selected features
df_selected_feture_train = df_train[["clean_title", "image_url", "2_way_label"]]
df_selected_feture_test = df_test[["clean_title", "image_url", "2_way_label"]]

# Define full original count per split
n_train = df_selected_feture_train.shape[0]
n_test = df_selected_feture_test.shape[0]
n_total = n_train + n_test

# Print original split ratios
split_ratios = {
    'train': n_train / n_total,
    'test': n_test / n_total,
}
print("Original split ratios:", split_ratios)

# Total you want after downsampling
target_total = 10000  # or any number you choose
target_split_sizes = {k: int(v * target_total) for k, v in split_ratios.items()}
print("Target split sizes:", target_split_sizes)

# Helper function to sample one split with class balance
def sample_split(df, target_size, label_col='2_way_label'):
    label_counts = df[label_col].value_counts()
    fractions = label_counts / label_counts.sum()
    target_per_class = (fractions * target_size).astype(int)

    print("  Class fractions:", fractions.to_dict())
    print("  Target per class:", target_per_class.to_dict())

    parts = []
    for label in target_per_class.index:
        part = df[df[label_col] == label].sample(n=target_per_class[label], random_state=42)
        parts.append(part)

    return pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

# Now generate each split
df_small_train = sample_split(df_selected_feture_train, target_split_sizes['train'])
df_small_test = sample_split(df_selected_feture_test, target_split_sizes['test'])

# Save each
os.makedirs("output", exist_ok=True)

df_small_train.to_csv('output/fakeddit_train.tsv', sep='\t', index=False)
df_small_test.to_csv('output/fakeddit_test.tsv', sep='\t', index=False)

print("✅ All reduced splits saved successfully!")


In [ ]:
df_small_train.shape

In [ ]:
df_small_test.shape

In [ ]:
df_small_train.head()

In [ ]:
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO



In [ ]:
def install(df, path):
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        if pd.notna(row['image_url']):
            try:
                response = requests.get(row['image_url'], timeout=5)
                response.raise_for_status()  # Raise HTTPError for bad status codes

                # Try to open and verify the image
                img = Image.open(BytesIO(response.content))
                img.verify()

                os.makedirs(f'images/{path}', exist_ok=True)
                with open(f'images/{path}/{row["id"]}.jpg', 'wb') as handler:
                    handler.write(response.content)

            except Exception as e:
                print(f"Failed to download: {row['image_url']}, reason: {e}")

In [ ]:
df_small_train.columns

In [ ]:
df_small_test.columns

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class MultimodalDataset(Dataset):
    def __init__(self, df, text_column='clean_title', image_dir='images/train',
                 label_column='2_way_label', transform=None):
        self.texts = df[text_column].astype(str).tolist()
        self.image_ids = df['id'].tolist()
        self.labels = df[label_column].astype(int).tolist()
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        image_path = os.path.join(self.image_dir, f"{self.image_ids[idx]}.jpg")
        try:
            image = Image.open(image_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), color='white')  # fallback dummy image

        if self.transform:
            image = self.transform(image)

        return text, image, label


In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # converts PIL image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


In [ ]:
import os

def extract_image_id(url):
    try:
        filename = os.path.basename(url)            # get last part of URL
        filename = filename.split('?')[0]           # remove query params
        file_id = os.path.splitext(filename)[0]     # remove .jpg/.png/etc
        return file_id
    except:
        return None

# Apply to all rows
df_small_train['id'] = df_small_train['image_url'].apply(extract_image_id)
df_small_test['id'] = df_small_test['image_url'].apply(extract_image_id)


In [ ]:
df_small_train.columns

In [ ]:
df_small_train.id.isna().sum()

In [ ]:
df_small_test.id.isna().sum()

In [ ]:
df_small_train=df_small_train.dropna()
df_small_test=df_small_test.dropna()

In [ ]:
df_small_train.id.isna().sum()

In [ ]:
df_small_train.head()

In [ ]:
train_dataset = MultimodalDataset(df_small_train, image_dir='images/train', transform=transform)
test_dataset  = MultimodalDataset(df_small_test, image_dir='images/test', transform=transform)

In [ ]:
# def get_sbert_embeddings(dataset, sbert_model, device):
#     embeddings = []
#     labels = []

#     sbert_model = sbert_model.to(device)
#     sbert_model.eval()

#     loader = DataLoader(dataset, batch_size=16, shuffle=False)

#     with torch.no_grad():
#         for batch in loader:
#             batch_embeddings = batch['embedding'].cpu().detach().numpy()
#             embeddings.extend(batch_embeddings)
#             labels.extend(batch['labels'].cpu().numpy())

#     return np.array(embeddings), np.array(labels)

In [ ]:
# def generate_sbert_embeddings(df_path, model_name='sentence-transformers/all-MiniLM-L6-v2'):
#     # Chargement des données
#     df = pd.read_csv(df_path)
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#     # Split des données
#     df_train, df_test = train_test_split(df, train_size=0.8, random_state=42)

#     # Chargement du modèle BERT
#     sbert_model = SentenceTransformer(model_name)

#     # Génération des embeddings
#     print("Génération des embeddings pour l'ensemble d'entraînement")
#     X_train, y_train = get_sbert_embeddings(SubjectDataset(df_train , sbert_model), sbert_model, device)

#     print("Génération des embeddings pour l'ensemble de test")
#     X_test, y_test = get_sbert_embeddings(SubjectDataset(df_test , sbert_model), sbert_model, device)

#     # Sauvegarde des embeddings
#     np.save('X_train_sbert_embeddings.npy', X_train)
#     np.save('y_train_sbert_labels.npy', y_train)
#     np.save('X_test_sbert_embeddings.npy', X_test)
#     np.save('y_test_sbert_labels.npy', y_test)

#     # Sauvegarder le modèle BERT
#     sbert_model.save('sbert_model')
#     return X_train, y_train, X_test, y_test ,sbert_model

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer

class SbertEmbeddingForText(nn.Module):
    def __init__(self, dataset, sbert_model, device, batch_size=16):
        super().__init__()
        self.texts = [sample[0] for sample in dataset]  # Extract text from (text, image, label)
        self.model = sbert_model
        self.device = device
        self.batch_size = batch_size

    def get_embedding(self):
        self.model.to(self.device)
        self.model.eval()

        dataloader = DataLoader(self.texts, batch_size=self.batch_size, shuffle=False)
        embeddings = []

        with torch.no_grad():
            for batch in dataloader:
                emb = self.model.encode(batch, convert_to_tensor=True, device=self.device)
                embeddings.append(emb)

        return torch.cat(embeddings, dim=0)


In [ ]:
# from sentence_transformers import SentenceTransformer
# import torch

# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# embedder = SbertEmbeddingForText(dataset=test_dataset, sbert_model=sbert_model, device=device)

# text_embeddings = embedder.get_embedding()

# print("✅ SBERT text embeddings generated!")
# print("Shape:", text_embeddings.shape)
# print("Example vector (first row):", text_embeddings[0])


In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
import os
from torch import nn

class EncodeImageFRCNN(nn.Module):
    def __init__(self, device='cpu', score_threshold=0.7, max_regions=10):
        super().__init__()
        self.device = device
        self.score_threshold = score_threshold
        self.max_regions = max_regions

        # Load pretrained Faster R-CNN model
        self.model = fasterrcnn_resnet50_fpn(pretrained=True)
        self.model.to(device)
        self.model.eval()

        # We'll use the backbone + ROI heads manually
        self.backbone = self.model.backbone
        self.rpn = self.model.rpn
        self.roi_heads = self.model.roi_heads
        self.transform = self.model.transform

    def forward(self, image_path):
        # Load and preprocess image
        image = Image.open(image_path).convert("RGB")
        image_tensor = F.to_tensor(image).to(self.device)

        # Faster R-CNN expects a list of images
        inputs = [image_tensor]
        # Apply transform (resizing, normalization, batching)
        processed_inputs, _ = self.transform(inputs)

        with torch.no_grad():
            # 1. Extract backbone features
            features = self.backbone(processed_inputs.tensors)

            # 2. Generate proposals
            proposals, _ = self.rpn(processed_inputs, features)

            # 3. Use ROI heads to get box features (not class predictions)
            box_features = self.roi_heads.box_roi_pool(
                features, proposals, processed_inputs.image_sizes
            )
            # 4. Feed pooled features into the head
            box_features = self.roi_heads.box_head(box_features)  # -> (N_regions, 1024)

        # Optionally limit by region count
        if box_features.shape[0] > self.max_regions:
            box_features = box_features[:self.max_regions]

        return box_features  # tensor of shape (N_regions, 1024)


In [ ]:
install(df_small_train, path='train')

In [ ]:
import shutil
import os

test_folder = '/kaggle/working/images/test'

# Delete the folder and recreate it
if os.path.exists(test_folder):
    shutil.rmtree(test_folder)
os.makedirs(test_folder, exist_ok=True)

print("✅ Cleared images/test/ folder.")

install(df_small_test, path='test')


In [ ]:
image_folder = '/content/images/test'
image_files = [f for f in os.listdir(image_folder) ]
encoder = EncodeImageFRCNN(device='cuda' if torch.cuda.is_available() else 'cpu')
# Sort if needed to ensure consistent order
image_files = sorted(image_files)[:3]  # first 3 only

for image_file in image_files:
    image_path = os.path.join(image_folder, image_file)
    features = encoder.forward(image_path)
    print(f"{image_file} → {features.shape}")


In [ ]:
image_files[:3]

In [ ]:
image_folder = '/content/images/test'
image_files = [f for f in os.listdir(image_folder) ]
encoder = EncodeImageFRCNN(device='cuda' if torch.cuda.is_available() else 'cpu')

image_files = sorted(image_files)[:3]  # first 3 only

for image_file in image_files:
    features = encoder(image_path)
    print("Extracted region features:", features.shape)  # e.g., torch.Size([10, 1024])


In [ ]:
# def extract_and_save_features(image_folder, feature_folder, encoder):
#     os.makedirs(feature_folder, exist_ok=True)
#     image_files = sorted([f for f in os.listdir(image_folder) if f.endswith('.jpg')])

#     for image_file in image_files:
#         image_path = os.path.join(image_folder, image_file)
#         try:
#             features = encoder(image_path)
#             save_path = os.path.join(feature_folder, image_file.replace('.jpg', '.pt'))
#             torch.save(features, save_path)
#         except Exception as e:
#             print(f" Failed on {image_file}: {e}")
import os
import torch
from tqdm import tqdm

def extract_and_save_features(image_folder, feature_folder, encoder):
    os.makedirs(feature_folder, exist_ok=True)
    image_files = sorted([f for f in os.listdir(image_folder) if f.endswith('.jpg')])

    for image_file in tqdm(image_files, desc=f"Processing {os.path.basename(image_folder)}"):
        image_path = os.path.join(image_folder, image_file)
        feature_path = os.path.join(feature_folder, image_file.replace('.jpg', '.pt'))

        # Skip if already exists and is valid
        if os.path.exists(feature_path):
            try:
                _ = torch.load(feature_path)
                continue  # skip if loadable
            except:
                os.remove(feature_path)  # corrupted, delete and retry

        try:
            features = encoder(image_path)  # Run FRCNN
            torch.save(features, feature_path)
        except Exception as e:
            print(f"❌ Failed on {image_file}: {e}")



In [ ]:
encoder = EncodeImageFRCNN(device='cuda' if torch.cuda.is_available() else 'cpu')
extract_and_save_features('/content/images/train', '/content/features/features_train', encoder)
extract_and_save_features('/content/images/test', '/content/features/features_test', encoder)


In [ ]:
# folder = "/kaggle/working/features/features_test"
# for file in os.listdir(folder):
#     if file.endswith(".pt"):
#         path = os.path.join(folder, file)
#         try:
#             torch.load(path)
#         except:
#             print(f"Deleting corrupted file: {file}")
#             os.remove(path)


In [ ]:
# `extract_and_save_features('/kaggle/working/images/test', '/kaggle/working/features/features_test', encoder)

In [ ]:
# from sentence_transformers import SentenceTransformer
# import torch

# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# embedder = SbertEmbeddingForText(dataset=test_dataset, sbert_model=sbert_model, device=device)

# text_embeddings = embedder.get_embedding()

# print("✅ SBERT text embeddings generated!")
# print("Shape:", text_embeddings.shape)
# print("Example vector (first row):", text_embeddings[0])

In [ ]:

# def embed_and_save_text(dataset, sbert_model, device, path_label, batch_size=16):
#     print(f"Embedding and saving text features for: {path_label}")

#     save_dir = f'/kaggle/working/text_features/text_features_{path_label}'
#     os.makedirs(save_dir, exist_ok=True)



# sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# embedder = SbertEmbeddingForText(dataset=test_dataset, sbert_model=sbert_model, device=device)

# text_embeddings = embedder.get_embedding()
#     if len(ids) != len(all_embeddings):
#         raise ValueError("Mismatch between sample IDs and embeddings.")

#     for i, sample_id in enumerate(ids):
#         try:
#             torch.save(
#                 {'text_entities': all_embeddings[i]},
#                 os.path.join(save_dir, f"{sample_id}.pt")
#             )
#         except Exception as e:
#             print(f"❌ Failed to save {sample_id}: {e}")

#     print(f"✅ Done! Saved {len(all_embeddings)} text embeddings to {save_dir}")


In [ ]:
# def embed_and_save_text(dataset, sbert_model, device, path_label, batch_size=16):
#     print(f" Embedding and saving text features for: {path_label}")

#     save_dir = f'/content/text_features/text_features_{path_label}'
#     os.makedirs(save_dir, exist_ok=True)

#     # Create SBERT embedder
#     embedder = SbertEmbeddingForText(dataset=dataset, sbert_model=sbert_model, device=device, batch_size=batch_size)
#     text_embeddings = embedder.get_embedding()

#     # Get image IDs from dataset
#     ids = dataset.image_ids

#     if len(ids) != len(text_embeddings):
#         raise ValueError("Mismatch between sample IDs and embeddings.")

#     for i, sample_id in enumerate(ids):
#         try:
#             torch.save(
#                 {'text_entities': text_embeddings[i]},
#                 os.path.join(save_dir, f"{sample_id}.pt")
#             )
#         except Exception as e:
#             print(f" Failed to save {sample_id}: {e}")

#     print(f"✅ Done! Saved {len(text_embeddings)} text embeddings to {save_dir}")


In [ ]:
# !rm -rf /kaggle/working/text_features
# !rm -rf /kaggle/working/__pycache__
# !rm -rf /kaggle/temp/*


In [ ]:
# !du -h /kaggle/working --max-depth=2 | sort -hr

In [ ]:
# !rm -rf /kaggle/working/images

In [ ]:
# import os
# import gc
# import torch
# from sentence_transformers import SentenceTransformer, util
# import numpy as np
# from tqdm import tqdm

# def embed_and_save_text(dataset, sbert_model, device, path_label='test', batch_size=16):
#     save_dir = f'/content/text_features/text_features_{path_label}'
#     os.makedirs(save_dir, exist_ok=True)

#     model = SentenceTransformer(sbert_model).to(device)

#     all_embeddings = []

#     for i in tqdm(range(0, len(dataset), batch_size), desc=f'Embedding {path_label} set'):
#         batch = dataset[i:i+batch_size]
#         texts = [example['clean_title'] for example in batch]
#         embeddings = model.encode(texts, convert_to_tensor=True, device=device).cpu().numpy()
#         all_embeddings.append(embeddings)


#     # Save all embeddings to a single file
#     output_path = os.path.join(save_dir, 'embeddings.npy')
#     np.save(output_path, np.vstack(all_embeddings))
# os.makedirs('/content/text_features', exist_ok=True)

# # Device selection
# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# # Replace with your actual datasets and model name
# # Make sure `test_dataset` and `train_dataset` are lists of dicts with key 'text'
# embed_and_save_text(train_dataset, 'all-MiniLM-L6-v2', device, path_label='train')
# embed_and_save_text(test_dataset, 'all-MiniLM-L6-v2', device, path_label='test')


In [ ]:
 df_small_train.head()

In [ ]:
import os, gc, torch, numpy as np
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def embed_and_save_text(dataset,
                        sbert_model='all-MiniLM-L6-v2',
                        device='cpu',
                        path_label='train',
                        batch_size=32,
                        num_workers=2):

    save_dir = f'/content/text_features/text_features_{path_label}'
    os.makedirs(save_dir, exist_ok=True)

    model = SentenceTransformer(sbert_model).to(device)

    loader = DataLoader(dataset,
                        batch_size=batch_size,
                        shuffle=False,
                        num_workers=num_workers,
                        collate_fn=lambda batch: list(zip(*batch)))  # keeps each element as a tuple column

    all_embs = []

    for texts, _, _ in tqdm(loader, desc=f'Embedding {path_label} set'):
        embs = model.encode(list(texts),
                            convert_to_tensor=True,
                            device=device).cpu().numpy()
        all_embs.append(embs)

        del embs
        gc.collect()

    np.save(os.path.join(save_dir, 'embeddings.npy'),
            np.vstack(all_embs))
    print(f'✅ Saved embeddings to {save_dir}/embeddings.npy  '
          f'({np.vstack(all_embs).shape[0]} rows)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

embed_and_save_text(train_dataset, device=device, path_label='train')
embed_and_save_text(test_dataset,  device=device, path_label='test')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiModalFusionLayer(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super(MultiModalFusionLayer, self).__init__()
        self.text_to_image_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.image_to_text_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm_text = nn.LayerNorm(embed_dim)
        self.norm_image = nn.LayerNorm(embed_dim)

    def forward(self, text_embeds, image_embeds):
        """
        text_embeds:  [B, M, D]  (M = number of tokens or sentences)
        image_embeds: [B, N, D]  (N = number of regions or patches)
        """
        # --- Text attending to Image ---
        text_attn_output, _ = self.text_to_image_attn(
            query=text_embeds, key=image_embeds, value=image_embeds
        )
        text_updated = self.norm_text(text_embeds + text_attn_output)

        # --- Image attending to Text ---
        image_attn_output, _ = self.image_to_text_attn(
            query=image_embeds, key=text_embeds, value=text_embeds
        )
        image_updated = self.norm_image(image_embeds + image_attn_output)

        return text_updated, image_updated


In [ ]:
import os

def filter_df_by_existing_features(df, feature_dir):
    valid_ids = []
    for id_ in df['id']:
        path = os.path.join(feature_dir, f'{id_}.pt')
        if os.path.exists(path):
            valid_ids.append(id_)
    return df[df['id'].isin(valid_ids)].reset_index(drop=True)

# Clean the training DataFrame
df_small_train = filter_df_by_existing_features(
    df_small_train,
    feature_dir='/content/features/features_train'
)


In [ ]:
import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class SimpleFusionDataset(Dataset):
    def __init__(self, df, text_npy, img_dir, label_col='2_way_label'):
        self.ids    = df['id'].tolist()
        self.labels = df[label_col].tolist()
        self.text   = np.load(text_npy)          # shape [N, 384]
        self.imgdir = img_dir

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        tid  = self.ids[idx]
        y    = torch.tensor(self.labels[idx]).long()

        txt  = torch.tensor(self.text[idx]).float()        # [384]
        img  = torch.load(os.path.join(self.imgdir, f'{tid}.pt')).float()  # [2048]

        return txt, img, y

# class SimpleFusionNet(nn.Module):
#     def __init__(self, txt_in=384, img_in=2048, hid=512, out=2):
#         super().__init__()
#         self.txt_proj = nn.Linear(txt_in, hid)
#         self.img_proj = nn.Linear(img_in, hid)
#         self.classifier = nn.Sequential(
#             nn.ReLU(),
#             nn.Linear(hid*2, hid),
#             nn.ReLU(),
#             nn.Linear(hid, out)
#         )

#     def forward(self, txt, img):
#       print('>> input txt:', txt.shape)
#       print('>> input img:', img.shape)

#       t = self.txt_proj(txt)
#       v = self.img_proj(img)

#       print('>> projected txt:', t.shape)
#       print('>> projected img:', v.shape)

#       x = torch.cat([t, v], dim=1)
#       print('>> concat:', x.shape)

#       return self.classifier(x)

class SimpleFusionNet(nn.Module):
    def __init__(self, txt_in=384, img_in=1024, hid=512, out=2):
        super().__init__()
        self.txt_proj = nn.Linear(txt_in, hid)        # 384 → 512
        self.img_proj = nn.Linear(img_in, hid)        # 1024 → 512  (applied to each patch)

        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(hid * 2, hid),  # 512(text) + 512(image) = 1024
            nn.ReLU(),
            nn.Linear(hid, out)
        )

    def forward(self, txt, img):
        """
        txt : [B, 384]
        img : [B, 10, 1024]   (10 visual regions)
        """
        t = self.txt_proj(txt)                 # [B, 512]

        v_seq = self.img_proj(img)             # [B, 10, 512]
        v     = v_seq.mean(dim=1)              # [B, 512]  simple average-pool

        x = torch.cat([t, v], dim=1)           # [B, 1024]
        return self.classifier(x)              # [B, 2]

# ----------  Training skeleton  ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_ds = SimpleFusionDataset(df_small_train,
                               '/content/text_features/text_features_train/embeddings.npy',
                               '/content/features/features_train')
loader = DataLoader(train_ds, batch_size=64, shuffle=True)

model = SimpleFusionNet().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()

for epoch in range(3):                       # tiny demo loop
    for txt, img, y in loader:
        txt, img, y = txt.to(device), img.to(device), y.to(device)
        logits = model(txt, img)
        loss   = lossf(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f'Epoch {epoch+1}: loss = {loss.item():.4f}')


In [ ]:
import os

def filter_df_by_existing_features(df, feature_dir):
    valid_ids = []
    for id_ in df['id']:
        path = os.path.join(feature_dir, f'{id_}.pt')
        if os.path.exists(path):
            valid_ids.append(id_)
    return df[df['id'].isin(valid_ids)].reset_index(drop=True)

# Clean the training DataFrame
df_small_test = filter_df_by_existing_features(
    df_small_test,
    feature_dir='/content/features/features_test'
)


In [ ]:
test_ds = SimpleFusionDataset(
    df=df_small_test,
    text_npy='/content/text_features/text_features_test/embeddings.npy',
    img_dir='/content/features/features_test'
)

test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)


In [ ]:
model.eval()  # disable dropout, etc.
all_preds = []
all_labels = []

with torch.no_grad():
    for txt, img, y in test_loader:
        txt, img = txt.to(device), img.to(device)
        logits = model(txt, img)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("✅ Accuracy:", accuracy_score(all_labels, all_preds))
print("✅ F1 Score:", f1_score(all_labels, all_preds))
print("\n📊 Classification Report:\n", classification_report(all_labels, all_preds))


In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

img = Image.open(image_path)
plt.imshow(img)



In [ ]:
caption = "this is me "

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

sbert = SentenceTransformer('all-MiniLM-L6-v2')
text_feat = sbert.encode([caption], convert_to_tensor=True)

In [ ]:
from PIL import Image
import torchvision.transforms as T

# Your feature extractor (replace this with your actual function)
def extract_image_features(image_path):
    image = Image.open(image_path).convert('RGB')

    # Resize and normalize (adjust to match your training transform)
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225])
    ])
    image_tensor = transform(image).unsqueeze(0)  # [1, 3, H, W]

    # Placeholder dummy features: you must replace this
    dummy_features = torch.randn(1, 10, 1024)  # shape: [1, 10, 1024]
    return dummy_features

# Get the filename
image_path = list(uploaded.keys())[0]
image_feat = extract_image_features(image_path)  # [1, 10, 1024]


In [ ]:
# Project text to 512 and pool image to 512
text_feat = text_feat.to(device)             # [1, 384]
image_feat = image_feat.to(device)           # [1, 10, 1024]

# Project text: [1, 384] → [1, 512]
text_proj = model.txt_proj(text_feat)

# Project image patches and pool: [1, 10, 1024] → [1, 512]
img_proj = model.img_proj(image_feat)        # [1, 10, 512]
img_pooled = img_proj.mean(dim=1)            # [1, 512]

# Concatenate and classify
x = torch.cat([text_proj, img_pooled], dim=1)   # [1, 1024]
logits = model.classifier(x)                    # [1, 2]
pred = torch.argmax(logits, dim=1).item()

label_map = {0: "REAL", 1: "FAKE"}
print("🧠 Prediction:", label_map[pred])


In [ ]:
torch.save(model, 'multimodal_model.pt')


In [ ]:
from google.colab import files
files.download('multimodal_model.pt')  # or 'multimodal_model_weights.pt'


In [ ]:
model = torch.load('multimodal_model.pt', weights_only=False)
model.eval()


In [ ]:
# !zip -r multimodal_backup.zip /content/features /content/text_features \
# /content/*.jpg /content/*.pt


In [ ]:
# from google.colab import files
# files.download("multimodal_backup.zip")
